# Credit Risk Decisioning System

End-to-end recruiter-grade workflow with four modeling tracks (LazyPredict, Manual Top-3, FLAML, PyCaret).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_prep import load_home_credit_data, build_customer_level_table, temporal_leakage_checks, stratified_split
from src.features import build_preprocessor, prepare_model_inputs
from src.modeling import (
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
    assign_decision_band,
    save_inference_bundle,
)
from src.evaluation import build_leaderboard, rerank_with_business_weights

SEED = 42
PROJECT_NAME = 'credit-risk-decisioning-system'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'home_credit'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Goal: reduce default risk while preserving approval volume.

Success criteria:
- Strong PR-AUC (class imbalance aware)
- Stable ROC-AUC and Brier calibration
- Actionable approve/review/reject policy
- Deployable scoring and monitoring hooks

## 2) Dataset Access and Data Dictionary

Dataset: Home Credit Default Risk (Kaggle).

Primary table:
- `application_train.csv`

Auxiliary tables used for aggregated customer signals:
- `bureau.csv`
- `previous_application.csv`
- `POS_CASH_balance.csv`
- `installments_payments.csv`
- `credit_card_balance.csv`

In [2]:
tables = load_home_credit_data(RAW_DIR, sample_frac=0.05, random_state=SEED)
{k: v.shape for k, v in tables.items()}

{'application_train': (15376, 122),
 'bureau': (72878, 17),
 'previous_application': (69999, 37),
 'POS_CASH_balance': (10001358, 8),
 'installments_payments': (13605401, 8),
 'credit_card_balance': (3840312, 23)}

## 3) Data Cleaning and Leakage Checks

- Build customer-level table from auxiliary aggregates
- Remove leak-prone or near-empty fields
- Validate target integrity before split

In [3]:
customer_df = build_customer_level_table(tables)
customer_df = temporal_leakage_checks(customer_df)
customer_df['TARGET'].value_counts(normalize=True)

TARGET
0    0.918444
1    0.081556
Name: proportion, dtype: float64

## 4) Feature Engineering

- Numeric aggregation features from auxiliary tables
- Categorical one-hot encoding
- Median/mode imputations and scaling for numeric stability

In [4]:
train_df, holdout_df = stratified_split(customer_df, target_col='TARGET', random_state=SEED)
preprocessor = build_preprocessor(train_df.drop(columns=['TARGET']))
X_train, X_holdout, y_train, y_holdout = prepare_model_inputs(
    train_df, holdout_df, target_col='TARGET', preprocessor=preprocessor
)
X_train.shape, X_holdout.shape

((12300, 371), (3076, 371))

## 5) Validation Strategy

- Stratified train/holdout split
- Track-specific CV where applicable (manual, FLAML, PyCaret)
- Primary metric: PR-AUC
- Secondary metric: ROC-AUC
- Tertiary metric: Brier score

In [5]:
summary = pd.DataFrame({
    'split': ['train', 'holdout'],
    'rows': [len(train_df), len(holdout_df)],
    'target_rate': [train_df['TARGET'].mean(), holdout_df['TARGET'].mean()]
})
summary

,split,rows,target_rate
0,train,12300,0.081545
1,holdout,3076,0.081599


## 6) Track 1: LazyPredict Discovery Lab

LazyPredict runs on the post-preprocessing matrix to identify promising model families quickly.

In [6]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

,Model,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
0,CatBoostClassifier,0.9200,0.502283,0.743815,0.882443,5.400215
1,LinearDiscriminantAnalysis,0.9120,0.541304,0.726220,0.889607,0.243510
2,RidgeClassifier,0.9196,0.499783,0.720967,0.881467,0.086818
3,RidgeClassifierCV,0.9196,0.499783,0.719707,0.881467,0.225804
4,CalibratedClassifierCV,0.9200,0.500000,0.706965,0.881667,89.504110
5,XGBClassifier,0.9172,0.516739,0.704722,0.885874,1.110887
6,LinearSVC,0.9164,0.502609,0.702141,0.881335,4.576482
7,LGBMClassifier,0.9184,0.505978,0.701304,0.883104,0.483222
8,LogisticRegression,0.9160,0.529783,0.695922,0.888841,0.246563
9,AdaBoostClassifier,0.9160,0.527500,0.682803,0.888259,3.450731


## 7) Selection of Top 3 Eligible Models

Eligibility filters applied:
- stability (Brier not extreme)
- speed and practicality
- interpretability fit for risk use-case

Only eligible top 3 families move into manual engineering.

In [7]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table, X_train, y_train, X_holdout, y_holdout, random_state=SEED
)
eligible_table, top3_families

/home/ahmad/AI/Projects/credit-risk-decisioning-system/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


(                   lazy_model               family    pr_auc   roc_auc  \
 0          CatBoostClassifier             catboost  0.262586  0.752329   
 1               XGBClassifier              xgboost  0.226912  0.752236   
 2  LinearDiscriminantAnalysis  logistic_regression  0.221028  0.729195   
 
       brier  train_time_sec interpretability  eligible eligibility_note  
 0  0.068493        8.561943       medium-low      True         eligible  
 1  0.071213       69.346832       medium-low      True         eligible  
 2  0.197666       11.593717             high      True         eligible  ,
 ['catboost', 'logistic_regression', 'xgboost'])

## 8) Track 2: Manual Engineering Lab

For each selected family:
- explicit training code
- calibration where relevant
- threshold optimization
- decision-band assignment

In [8]:
manual_results, manual_models = run_manual_engineering_track(
    top3_families=top3_families,
    X_train=X_train, y_train=y_train,
    X_holdout=X_holdout, y_holdout=y_holdout,
    random_state=SEED,
)
manual_results[['model_name', 'pr_auc', 'roc_auc', 'brier', 'optimized_threshold', 'policy_utility']]

/home/ahmad/AI/Projects/credit-risk-decisioning-system/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model_name,pr_auc,roc_auc,brier,optimized_threshold,policy_utility
0,catboost,0.271494,0.762344,0.068580,0.11,-615.0
1,xgboost,0.240332,0.762309,0.070456,0.10,-611.0
2,logistic_regression,0.221028,0.729195,0.197666,0.63,-849.0


In [9]:
best_manual_idx = manual_results['pr_auc'].idxmax()
best_manual_row = manual_results.loc[best_manual_idx]
bands = assign_decision_band(
    best_manual_row['holdout_scores'],
    approve_threshold=0.25,
    reject_threshold=0.65,
)
pd.Series(bands).value_counts(normalize=True).rename('share')

approve          0.970741
manual_review    0.028609
reject           0.000650
Name: share, dtype: float64

## 9) Track 3: FLAML Optimization Lab

FLAML is run with explicit budget and PR-AUC objective to search for budget-aware challengers.

In [10]:
flaml_result = run_flaml_track(
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    y_holdout=y_holdout,
    time_budget=240,
    random_state=SEED,
)
flaml_result

[flaml.automl.logger: 06-11 18:29:23] {2375} INFO - task = classification


[flaml.automl.logger: 06-11 18:29:23] {2386} INFO - Evaluation method: cv


[flaml.automl.logger: 06-11 18:29:23] {2489} INFO - Minimizing error metric: 1-ap


[flaml.automl.logger: 06-11 18:29:23] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'xgboost', 'rf', 'extra_tree', 'lrl1']


[flaml.automl.logger: 06-11 18:29:23] {2911} INFO - iteration 0, current learner lgbm


...

[flaml.automl.logger: 06-11 18:33:27] {3359} INFO - retrain lgbm for 3.3s


[flaml.automl.logger: 06-11 18:33:27] {3362} INFO - retrained model: LGBMClassifier(colsample_bytree=0.8913821757158379,
               learning_rate=0.19961122310826113, max_bin=31,
               min_child_samples=13, n_estimators=41, n_jobs=-1, num_leaves=5,
               reg_alpha=0.17967797142195352, reg_lambda=0.030313868202178677,
               verbose=-1)


[flaml.automl.logger: 06-11 18:33:27] {2636} INFO - fit succeeded


[flaml.automl.logger: 06-11 18:33:27] {2637} INFO - Time taken to find the best model: 119.48342084884644


{'model_name': 'lgbm',
 'library_source': 'flaml',
 'pr_auc': 0.24883879538861708,
 'roc_auc': 0.7512364700490076,
 'brier': 0.0686443159859727,
 'train_time_sec': 243.57422753600986,
 'infer_latency_ms': 3.0018923660585037,
 'p95_latency_ms': 6.029514454712626,
 'calibration_metric': 0.0686443159859727,
 'best_config': {'n_estimators': 41,
  'num_leaves': 5,
  'min_child_samples': 13,
  'learning_rate': 0.19961122310826113,
  'log_max_bin': 5,
  'colsample_bytree': 0.8913821757158379,
  'reg_alpha': 0.17967797142195352,
  'reg_lambda': 0.030313868202178677},
 'best_loss': 0.7800174557835158,
 'time_budget': 240,
 'interpretability_note': 'FLAML challenger under explicit budget'}

## 10) Track 4: PyCaret Experiment Lab

PyCaret orchestration for compare/tune/calibrate/finalize workflow and deployability check.

In [11]:
pycaret_result = run_pycaret_track(
    train_df=train_df,
    holdout_df=holdout_df,
    target_col='TARGET',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_credit_risk',
)
pycaret_result

{'model_name': 'pycaret_failed',
 'library_source': 'pycaret',
 'pr_auc': nan,
 'roc_auc': nan,
 'brier': nan,
 'train_time_sec': nan,
 'infer_latency_ms': nan,
 'p95_latency_ms': nan,
 'calibration_metric': nan,
 'interpretability_note': "PyCaret unavailable or failed: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=10, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')",
 'status': 'failed'}

## 11) Unified Leaderboard and Final Model Ranking

Includes LazyPredict top candidates, manual top 3, FLAML winner, PyCaret finalized model, and business baseline.

In [12]:
baseline_rows = [{
    'model_name': 'rule_based_income_ratio_baseline',
    'primary': 0.0,
    'secondary': 0.0,
    'tertiary': 0.25,
    'note': 'Business reference only; replace after scorecard benchmark run'
}]

leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    task_type='binary_classification',
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
    baseline_rows=baseline_rows,
    manual_model_objects=manual_models,
)
leaderboard = rerank_with_business_weights(leaderboard)
leaderboard.head(10)

,project_name,task_type,library_source,model_name,cv_metric_mean,cv_metric_std,holdout_primary_metric,holdout_secondary_metric,holdout_tertiary_metric,calibration_metric,train_time_sec,infer_latency_ms,p95_latency_ms,model_size_mb,retrain_time_sec,interpretability_note,rank_score,final_rank
0,credit-risk-decisioning-system,binary_classification,manual,catboost,NaN,NaN,0.271494,0.762344,0.068580,0.068580,27.838844,2.998693,3.778982,4.447565,27.838844,Manual catboost with explicit thresholds and c...,0.341777,1
1,credit-risk-decisioning-system,binary_classification,lazypredict,catboost,NaN,NaN,0.262586,0.752329,0.068493,0.068493,8.561943,NaN,NaN,NaN,NaN,eligible,0.332494,2
2,credit-risk-decisioning-system,binary_classification,flaml,lgbm,NaN,NaN,0.248839,0.751236,0.068644,0.068644,243.574228,3.001892,6.029514,NaN,243.574228,FLAML challenger under explicit budget,0.322514,3
3,credit-risk-decisioning-system,binary_classification,lazypredict,xgboost,NaN,NaN,0.226912,0.752236,0.071213,0.071213,69.346832,NaN,NaN,NaN,NaN,eligible,0.310795,4
4,credit-risk-decisioning-system,binary_classification,manual,logistic_regression,NaN,NaN,0.221028,0.729195,0.197666,0.197666,13.817003,0.073335,0.099667,0.003676,13.817003,Manual logistic_regression with explicit thres...,0.295021,5
5,credit-risk-decisioning-system,binary_classification,lazypredict,logistic_regression,NaN,NaN,0.221028,0.729195,0.197666,0.197666,11.593717,NaN,NaN,NaN,NaN,eligible,0.288858,6
6,credit-risk-decisioning-system,binary_classification,manual,xgboost,NaN,NaN,0.240332,0.762309,0.070456,0.070456,192.840122,27.408260,38.979984,4.268757,192.840122,Manual xgboost with explicit thresholds and ca...,0.277731,7
7,credit-risk-decisioning-system,binary_classification,baseline,rule_based_income_ratio_baseline,NaN,NaN,0.000000,0.000000,0.250000,NaN,NaN,NaN,NaN,NaN,NaN,Business reference only; replace after scoreca...,-0.031291,8
8,credit-risk-decisioning-system,binary_classification,pycaret,pycaret_failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PyCaret unavailable or failed: ('Pycaret only ...,-0.031291,9


In [13]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_credit_risk.csv', index=False)
leaderboard[['model_name', 'library_source', 'rank_score', 'final_rank']].head(10)

,model_name,library_source,rank_score,final_rank
0,catboost,manual,0.341777,1
1,catboost,lazypredict,0.332494,2
2,lgbm,flaml,0.322514,3
3,xgboost,lazypredict,0.310795,4
4,logistic_regression,manual,0.295021,5
5,logistic_regression,lazypredict,0.288858,6
6,xgboost,manual,0.277731,7
7,rule_based_income_ratio_baseline,baseline,-0.031291,8
8,pycaret_failed,pycaret,-0.031291,9


## 12) Business Recommendation

Document in this section after execution:
- Why rank-1 is preferred for production
- When rank-2 may be safer (stability/latency/interpretability)
- Which tradeoff prevented another strong model from winning

## 13) Inference / Deployment Path

Persist chosen model + preprocessor + thresholds for FastAPI scoring.

In [14]:
winner = leaderboard.sort_values('final_rank').iloc[0]['model_name']
if winner in manual_models:
    save_inference_bundle(
        model=manual_models[winner],
        preprocessor=preprocessor,
        output_dir=ARTIFACT_DIR,
        approve_threshold=0.25,
        reject_threshold=0.65,
    )
    print(f'Saved manual winner: {winner}')
else:
    print('Winner not in manual track; save selected production artifact separately.')

Saved manual winner: catboost


## 14) Monitoring / Drift / Retraining Plan

Recommended production checks:
- PR-AUC, ROC-AUC, Brier on rolling windows
- band distribution drift (approve/review/reject)
- score PSI and feature drift
- manual-review override rate
- retraining SLA and champion/challenger cadence

## 15) Limitations and Next Steps

- Add external bureau updates for recency-aware features
- Stress test fairness and reject-inference strategy
- Add explainability slices by segment and approval band
- Run seed stability reruns for top 3 finalists